# Riduzione della dimensionalità con PCA

La PCA, Principal Component Analysis, è una tecnica di riduzione della dimensionalità.

A differenza della feature selection, non sceglie alcune feature originali e ne scarta altre.

La PCA costruisce nuove feature, chiamate componenti principali, ottenute come combinazioni lineari delle feature originali.

L'obiettivo è passare da uno spazio con molte feature:

$$
d
$$

a uno spazio più piccolo:

$$
k < d
$$

conservando quanta più informazione possibile.

L'informazione viene misurata tramite la varianza.

Le prime componenti principali sono le direzioni lungo cui i dati variano di più.

## Idea geometrica

Se due feature sono correlate, significa che contengono informazione ridondante.

Per esempio, se al crescere di $x_1$ cresce anche $x_2$, i punti tendono a disporsi lungo una direzione obliqua.

La PCA cerca una nuova direzione, chiamata prima componente principale:

$$
PC1
$$

lungo cui i dati sono più sparpagliati.

Poi cerca una seconda direzione:

$$
PC2
$$

perpendicolare alla prima, che spiega la massima varianza residua.

Le componenti principali sono quindi:

```text
direzioni ortogonali
ordinate per varianza spiegata
```

Se scegliamo solo le prime $k$ componenti, comprimiamo il dataset mantenendo le direzioni più informative.

## Procedura della PCA

Sia:

$$
X \in \mathbb{R}^{n \times d}
$$

dove:

- $n$ è il numero di esempi;
- $d$ è il numero di feature.

La PCA segue questi passaggi:

1. centrare $X$, sottraendo la media di ogni feature;
2. calcolare la matrice di covarianza:

$$
C = \frac{1}{n-1}X^T X
$$

3. calcolare autovalori e autovettori di $C$;
4. ordinare gli autovettori in base agli autovalori decrescenti;
5. scegliere i primi $k$ autovettori;
6. costruire la matrice di proiezione $W$;
7. proiettare i dati:

$$
X_{PCA} = XW
$$

Il nuovo dataset avrà dimensione:

$$
n \times k
$$

## Dataset sintetico

Viene generato un dataset di classificazione binaria con molte feature.

Il dataset contiene:

```text
10000 esempi
100 feature
10 feature informative
2 classi
```

Il parametro:

```python
flip_y=0.1
```

introduce rumore nelle etichette.

Quindi una parte delle etichette può essere modificata casualmente.

Il problema serve a confrontare due casi:

```text
classificazione sulle feature originali
classificazione dopo riduzione con PCA
```

Nota: nel commento del codice è scritto "due feature", ma il valore effettivo è `n_features=100`.

In [9]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=10000,        # numero di campioni
    n_features=100,          # due feature per visualizzare facilmente
    n_informative=10,       # entrambe informative
    n_redundant=0,         # nessuna feature ridondante
    n_repeated=0,          # nessuna feature duplicata
    n_classes=2,           # classificazione binaria
    n_clusters_per_class=1,# un solo cluster per classe
    class_sep=1.0,         # maggiore separazione tra le classi
    flip_y=0.1,              # nessun rumore nelle etichette
    random_state=30,
)

## Suddivisione training/test

Il dataset viene diviso in:

```text
training set
test set
```

Il training set viene usato per addestrare il modello e per calcolare la PCA.

Il test set viene usato solo per valutare le prestazioni finali.

La variabile:

```python
n = X_train.shape[0]
```

salva il numero di esempi nel training set.

Questo valore serve per calcolare la matrice di covarianza:

$$
C = \frac{1}{n-1}X^T X
$$

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

n = X_train.shape[0]

## Modello sulle feature originali

Prima viene addestrato un modello usando tutte le feature originali.

Il dataset ha:

```text
100 feature
```

Il classificatore usato è una regressione logistica.

Questa accuracy serve come riferimento.

Dopo applicheremo la PCA e confronteremo le prestazioni usando meno dimensioni.

In [3]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = np.mean(y_test == y_pred)

print(accuracy)

0.8703333333333333


## Applicazione PCA

Il primo passaggio della PCA è centrare i dati.

Centrare significa sottrarre a ogni feature la propria media calcolata sul training set:

$$
X_{train,c} = X_{train} - \mu
$$

dove:

$$
\mu
$$

è il vettore delle medie delle feature.

Dopo la centratura, ogni colonna del training set ha media circa zero.

Questo passaggio è necessario perché la PCA studia la varianza rispetto alla media.

In [4]:
# matrice di covarianza
X_train_c = X_train - X_train.mean(axis=0)

C = (X_train_c.T @ X_train_c) / (n - 1)

## Matrice di covarianza

La matrice di covarianza è:

$$
C = \frac{1}{n-1}X^TX
$$

dove $X$ è il training set centrato.

Se $X$ ha dimensione:

$$
n \times d
$$

allora:

$$
X^T X
$$

ha dimensione:

$$
d \times d
$$

Quindi anche la matrice di covarianza $C$ ha dimensione:

$$
d \times d
$$

Nel nostro caso:

```text
d = 100
```

quindi $C$ è una matrice:

$$
100 \times 100
$$

## Significato degli elementi della matrice

La matrice di covarianza ha questa forma:

$$
C =
\begin{bmatrix}
\mathrm{Var}(x_1) & \mathrm{Cov}(x_1, x_2) & \cdots & \mathrm{Cov}(x_1, x_d) \\
\mathrm{Cov}(x_2, x_1) & \mathrm{Var}(x_2) & \cdots & \mathrm{Cov}(x_2, x_d) \\
\vdots & \vdots & \ddots & \vdots \\
\mathrm{Cov}(x_d, x_1) & \mathrm{Cov}(x_d, x_2) & \cdots & \mathrm{Var}(x_d)
\end{bmatrix}
$$

Sulla diagonale ci sono le varianze delle singole feature.

Fuori dalla diagonale ci sono le covarianze tra coppie di feature.

La matrice è simmetrica perché:

$$
\mathrm{Cov}(x_i, x_j) = \mathrm{Cov}(x_j, x_i)
$$

## Varianza lungo una direzione

Sia:

$$
v \in \mathbb{R}^d
$$

una direzione nello spazio delle feature, con:

$$
\|v\| = 1
$$

La proiezione dei dati lungo la direzione $v$ è:

$$
Xv
$$

Questa operazione prende ogni esempio e lo proietta su un asse orientato secondo $v$.

La PCA cerca la direzione $v$ che massimizza la varianza della proiezione:

$$
\max_{\|v\| = 1} \mathrm{Var}(Xv)
$$

È noto che:

$$
\mathrm{Var}(Xv) = v^T C v
$$

Quindi il problema diventa:

$$
\max_{v \in \mathbb{R}^d, \|v\| = 1} v^T C v
$$

## Dimostrazione di $\mathrm{Var}(Xv) = v^T C v$

Assumiamo:

- $X \in \mathbb{R}^{n \times d}$ centrata;
- $v \in \mathbb{R}^{d}$;
- $C = \frac{1}{n-1}X^TX$.

La proiezione dei dati lungo $v$ è:

$$
z = Xv
$$

dove:

$$
z \in \mathbb{R}^{n}
$$

Ogni elemento di $z$ è:

$$
z_i = x_i^T v
$$

Poiché $X$ è centrata, anche $z$ è centrato.

Quindi:

$$
\mathrm{Var}(z) = \frac{1}{n-1}z^Tz
$$

Sostituendo $z = Xv$:

$$
\mathrm{Var}(Xv) = \frac{1}{n-1}(Xv)^T(Xv)
$$

Poiché:

$$
(Xv)^T = v^T X^T
$$

abbiamo:

$$
(Xv)^T(Xv) = v^T X^T X v
$$

Quindi:

$$
\mathrm{Var}(Xv)
=
\frac{1}{n-1}v^T X^T X v
$$

Dato che:

$$
C = \frac{1}{n-1}X^TX
$$

otteniamo:

$$
\mathrm{Var}(Xv) = v^T C v
$$

## Autovalori e autovettori della matrice di covarianza

La PCA cerca le direzioni che massimizzano la varianza.

Queste direzioni sono gli autovettori della matrice di covarianza $C$.

Gli autovalori associati indicano quanta varianza viene spiegata da ciascuna direzione.

Se:

$$
Cw = \lambda w
$$

allora $w$ è un autovettore e $\lambda$ è il suo autovalore.

Poiché gli autovettori sono normalizzati:

$$
\|w\| = 1
$$

si ha:

$$
w^T C w = \lambda
$$

Quindi l'autovalore rappresenta la varianza spiegata lungo la direzione dell'autovettore.

In [5]:
eigenvalues , eigenvectors = np.linalg.eigh(C)   # autovettori sulle colonnne

idx = np.argsort(eigenvalues)[::-1]

W = eigenvectors[:, idx[:20]]

X_train_pca = X_train_c @ W

## Ordinamento delle componenti principali

La funzione:

```python
np.linalg.eigh(C)
```

calcola autovalori e autovettori della matrice di covarianza.

Si usa `eigh` perché $C$ è simmetrica.

Gli autovettori sono restituiti nelle colonne della matrice `eigenvectors`.

La riga:

```python
idx = np.argsort(eigenvalues)[::-1]
```

ordina gli indici degli autovalori dal più grande al più piccolo.

Poi:

```python
W = eigenvectors[:, idx[:20]]
```

seleziona i 20 autovettori associati ai 20 autovalori più grandi.

Quindi:

```text
feature originali: 100
componenti PCA: 20
```

La proiezione finale è:

```python
X_train_pca = X_train_c @ W
```

che produce un nuovo training set con 20 dimensioni.

## Massimizzazione della varianza

La PCA risolve il problema:

$$
\max_{v \in \mathbb{R}^d, \|v\| = 1} v^T C v
$$

La soluzione è l'autovettore di $C$ con autovalore massimo.

Se $w$ è un autovettore:

$$
Cw = \lambda w
$$

allora:

$$
w^T C w = w^T \lambda w
$$

$$
w^T C w = \lambda w^T w
$$

Poiché:

$$
\|w\| = 1
$$

si ottiene:

$$
w^T C w = \lambda
$$

Quindi massimizzare la varianza significa scegliere l'autovettore con autovalore più grande.

Gli autovettori successivi spiegano la massima varianza residua, mantenendo l'ortogonalità rispetto ai precedenti.

## Applicazione alla classificazione

Dopo la PCA, il modello non viene più addestrato su tutte le feature originali.

Viene addestrato su:

```python
X_train_pca
```

che contiene solo 20 componenti principali.

Il test set deve essere trasformato nello stesso modo.

Attenzione: il test set va centrato usando la media del training set, non la propria media.

Per questo si usa:

```python
X_test - X_train.mean(axis=0)
```

e poi la stessa matrice di proiezione:

```python
W
```

La trasformazione è:

$$
X_{test,PCA} = (X_{test} - \mu_{train})W
$$

In [6]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train_pca, y_train)

X_test_pca = (X_test - X_train.mean(axis=0)) @ W

y_pred = model.predict(X_test_pca)

accuracy = np.mean(y_test == y_pred)

print(accuracy)

0.8476666666666667


## Autovalori e varianza totale

Gli autovalori indicano quanta varianza viene spiegata dalle rispettive componenti principali.

Stampare gli autovalori in ordine decrescente permette di vedere quali componenti spiegano più varianza.

La somma degli autovalori rappresenta la varianza totale del dataset:

$$
\sum_{j=1}^{d} \lambda_j
$$

Se le prime componenti hanno autovalori molto grandi rispetto alle successive, significa che molta informazione è concentrata in poche direzioni.

Questo giustifica la riduzione da 100 feature a 20 componenti principali.

In [7]:
for x in sorted(eigenvalues, reverse=True):
    print(x)

print(sum(eigenvalues))

7.112507339934967
6.958657791014027
5.1876732338898055
4.665342474801907
3.252707390808801
2.591512732801651
2.2094500430974153
2.0815385853746555
1.4640027268556193
1.240422242700828
1.2374259008784536
1.2129882187156211
1.1950708218879265
1.1874965199021579
1.1824855610690637
1.173585690880967
1.1655733564081174
1.1599817020902405
1.155884485519769
1.1517600623337025
1.1497569022107281
1.1449189903764168
1.136405356095912
1.1277505204371814
1.1215994290194595
1.1176062109489828
1.111054861642521
1.1062194991238614
1.1022282241031902
1.0995034564789898
1.0950445120467511
1.0896715317235206
1.0867785399070136
1.0798126259964795
1.0752063727982801
1.0712039951449976
1.0661823554394174
1.0643718793968135
1.057872871447647
1.0540333798595711
1.0461883758398696
1.0401022311036427
1.0381532345916684
1.0351911595959054
1.0336172371086392
1.0260361310294746
1.0223606483768628
1.0209334075057888
1.0198680612684807
1.0116484779452972
1.0098451059196647
1.0033358179437197
0.9963019009876012
0.99

## Varianza spiegata

Per capire quanta informazione viene conservata dalle prime $k$ componenti, si può calcolare:

$$
\frac{\sum_{j=1}^{k}\lambda_j}{\sum_{j=1}^{d}\lambda_j}
$$

Nel codice viene scelta:

```python
k = 20
```

quindi la quota di varianza spiegata dalle prime 20 componenti è:

$$
\frac{\lambda_1 + \lambda_2 + \cdots + \lambda_{20}}
{\lambda_1 + \lambda_2 + \cdots + \lambda_d}
$$

Questa quantità permette di valutare se la riduzione dimensionale conserva abbastanza informazione.

In [8]:
explained_variance_ratio = eigenvalues[idx] / np.sum(eigenvalues)

print("Varianza spiegata dalle prime 20 componenti:")
print(np.sum(explained_variance_ratio[:20]))

print("\nVarianza spiegata cumulativa:")
print(np.cumsum(explained_variance_ratio[:20]))

Varianza spiegata dalle prime 20 componenti:
0.38467956435981443

Varianza spiegata cumulativa:
[0.05631319 0.11140827 0.15248161 0.1894194  0.21517267 0.23569093
 0.25318423 0.26966478 0.28125601 0.29107703 0.30087434 0.31047816
 0.31994011 0.3293421  0.33870442 0.34799627 0.35722468 0.36640882
 0.37556052 0.38467956]


## Costo computazionale

Sia:

- $n$ il numero di esempi;
- $d$ il numero di feature originali;
- $k$ il numero di componenti mantenute.

La centratura del dataset ha costo:

$$
O(nd)
$$

Il calcolo della matrice di covarianza:

$$
C = X^TX
$$

ha costo:

$$
O(nd^2)
$$

La decomposizione agli autovalori della matrice $d \times d$ ha costo circa:

$$
O(d^3)
$$

La proiezione dei dati su $k$ componenti:

$$
XW
$$

ha costo:

$$
O(ndk)
$$

La PCA è quindi conveniente quando, dopo la trasformazione, si riduce molto il numero di feature usate dai modelli successivi.